# AquaRender — Kaggle Engine Notebook

This notebook boots a ComfyUI server with SDXL + watercolor LoRA + ControlNet
and exposes it via a Cloudflare Tunnel. Copy the printed URL into the
AquaRender Connect tab on your laptop.

**Setup:**
1. Settings → Accelerator → **GPU P100** (or T4)
2. Settings → Internet → **On**
3. Click **Run All**. Setup takes ~5 min the first time.
4. Look for the green `✅ AquaRender engine ready` banner — copy the URL.

⚠️  The tunnel URL changes every time you restart the notebook.


## 1. Detect GPU

In [ ]:
import subprocess

try:
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
    print("GPU detected:")
    print(out.decode())
except Exception:
    raise SystemExit(
        "❌ No GPU. Open Settings → Accelerator → GPU P100 (or T4) and re-run."
    )


## 2. Clone ComfyUI (pinned)

In [ ]:
import os
import pathlib
import subprocess

COMFY_DIR = pathlib.Path("/kaggle/working/ComfyUI")
COMFY_COMMIT = "b9d31a9b9b0bff58b85c9e34e62fa7b9b3b5d5c3"  # update with care

if not COMFY_DIR.exists():
    subprocess.check_call(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)])
subprocess.check_call(["git", "-C", str(COMFY_DIR), "fetch", "--depth=1", "origin", COMFY_COMMIT], stderr=subprocess.DEVNULL)
subprocess.check_call(["git", "-C", str(COMFY_DIR), "checkout", COMFY_COMMIT])
subprocess.check_call(["pip", "install", "--quiet", "-r", str(COMFY_DIR / "requirements.txt")])
print("ComfyUI ready at", COMFY_DIR)


## 3. Download SDXL Base 1.0 (~7 GB)

In [ ]:
import pathlib
import subprocess

ckpt_dir = pathlib.Path("/kaggle/working/ComfyUI/models/checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)
target = ckpt_dir / "sd_xl_base_1.0.safetensors"
if not target.exists():
    subprocess.check_call([
        "wget", "-q", "--show-progress", "-O", str(target),
        "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors",
    ])
print("Checkpoint:", target.stat().st_size / 1e9, "GB")


## 4. Download watercolor LoRA

In [ ]:
import pathlib
import subprocess

lora_dir = pathlib.Path("/kaggle/working/ComfyUI/models/loras")
lora_dir.mkdir(parents=True, exist_ok=True)
target = lora_dir / "watercolor_style_lora_sdxl.safetensors"
if not target.exists():
    subprocess.check_call([
        "wget", "-q", "--show-progress", "-O", str(target),
        "https://huggingface.co/ostris/watercolor_style_lora_sdxl/resolve/main/watercolor_style_lora_sdxl.safetensors",
    ])
print("LoRA:", target.stat().st_size / 1e6, "MB")


## 5. Download ControlNet (Lineart + Canny)

In [ ]:
import pathlib
import subprocess

cn_dir = pathlib.Path("/kaggle/working/ComfyUI/models/controlnet")
cn_dir.mkdir(parents=True, exist_ok=True)
LINEART = cn_dir / "diffusers_xl_lineart_full.safetensors"
CANNY = cn_dir / "diffusers_xl_canny_full.safetensors"
if not LINEART.exists():
    subprocess.check_call([
        "wget", "-q", "--show-progress", "-O", str(LINEART),
        "https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/diffusers_xl_lineart_full.safetensors",
    ])
if not CANNY.exists():
    subprocess.check_call([
        "wget", "-q", "--show-progress", "-O", str(CANNY),
        "https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/diffusers_xl_canny_full.safetensors",
    ])
print("ControlNets ready")


## 6. Symlink user-uploaded LoRA datasets

In [ ]:
import pathlib

input_root = pathlib.Path("/kaggle/input")
lora_dir = pathlib.Path("/kaggle/working/ComfyUI/models/loras")
linked = []
if input_root.exists():
    for ds in input_root.iterdir():
        if not ds.is_dir():
            continue
        for f in ds.rglob("*.safetensors"):
            link = lora_dir / f.name
            if link.exists():
                continue
            try:
                os.symlink(f, link)
                linked.append(f.name)
            except OSError:
                pass
print(f"Linked {len(linked)} LoRA(s) from /kaggle/input/: {linked}")


## 7. Install cloudflared

In [ ]:
import os
import pathlib
import subprocess

bin_path = pathlib.Path("/kaggle/working/cloudflared")
if not bin_path.exists():
    subprocess.check_call([
        "wget", "-q", "-O", str(bin_path),
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    ])
    os.chmod(bin_path, 0o755)
print("cloudflared:", bin_path)


## 8. Launch ComfyUI in the background

In [ ]:
import os
import pathlib
import subprocess
import time

LOG = pathlib.Path("/kaggle/working/comfyui.log")
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd="/kaggle/working/ComfyUI",
    stdout=open(LOG, "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid,
)
print("ComfyUI PID:", proc.pid)


## 9. Wait for ComfyUI to be ready

In [ ]:
import json
import urllib.request

deadline = time.time() + 240
while time.time() < deadline:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as r:
            stats = json.load(r)
            print("✅ ComfyUI ready —", stats.get("system", {}).get("comfyui_version", "unknown"))
            break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit("ComfyUI never became ready. Check /kaggle/working/comfyui.log")


## 10. Launch Cloudflare Tunnel & print URL

In [ ]:
import os
import pathlib
import re
import subprocess
import time

TUNNEL_LOG = pathlib.Path("/kaggle/working/cloudflared.log")
tunnel = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
    stdout=open(TUNNEL_LOG, "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid,
)

URL_RE = re.compile(r"https://[-\w]+\.trycloudflare\.com")
url = None
deadline = time.time() + 90
while time.time() < deadline and url is None:
    time.sleep(1)
    if TUNNEL_LOG.exists():
        text = TUNNEL_LOG.read_text(errors="ignore")
        m = URL_RE.search(text)
        if m:
            url = m.group(0)

if url is None:
    raise SystemExit("Could not parse tunnel URL. See /kaggle/working/cloudflared.log")

print("\n" + "=" * 64)
print("✅ AquaRender engine ready")
print("-" * 64)
print(f"Engine URL:  {url}")
print("Copy this URL into AquaRender's Connect tab.")
print("=" * 64)
print("⚠️  This URL changes every time the notebook restarts.")


## 11. Keep this cell running (session stays alive)

In [ ]:
import subprocess

subprocess.call(["tail", "-F", "/kaggle/working/comfyui.log"])
